# SICR and staging

Staging combines lifetime-PD deterioration, delinquency, the 30-DPD backstop, modification history and default.

In [1]:
from pathlib import Path
import json
import sqlite3
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, brier_score_loss, mean_absolute_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().resolve()
if not (ROOT / "config").exists():
    ROOT = ROOT.parent
DB = ROOT / "database" / "ifrs9_ecl.sqlite3"

def query(sql):
    with sqlite3.connect(DB) as connection:
        return pd.read_sql_query(sql, connection)

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")
plt.rcParams["figure.figsize"] = (9, 4)

Stage 3 is assigned first. Stage 2 is assigned when any configured SICR trigger applies. Remaining loans are Stage 1.

In [2]:
query('select * from stage_summary order by stage')

,stage,loans,gross_exposure,ecl,coverage_ratio,exposure_share,ecl_share
0,1,192,"28,490,378.8900","1,105.3745",0.0000,0.0658,0.0026
1,2,3261,"401,710,712.7500","387,650.6954",0.0010,0.9278,0.9143
2,3,18,"2,762,680.3300","35,245.7956",0.0128,0.0064,0.0831


In [3]:
query('''select stage, primary_stage_reason, count(*) loans,
sum(current_actual_upb) exposure
from reporting_date_portfolio
group by stage, primary_stage_reason
order by stage, loans desc''')

,stage,primary_stage_reason,loans,exposure
0,1,No SICR trigger,192,"28,490,378.8900"
1,2,Absolute lifetime-PD deterioration,1842,"260,677,013.2500"
2,2,Relative lifetime-PD deterioration,1161,"98,493,662.9400"
3,2,Prior default history,219,"36,837,850.2400"
4,2,30 DPD backstop,38,"5,545,623.3000"
5,2,Modification qualitative indicator,1,"156,563.0200"
6,3,Current default/credit-impaired,18,"2,762,680.3300"


Stage 1 uses defaults in the next 12 months. Stage 2 uses remaining lifetime. Stage 3 uses discounted recovery cash shortfall.